In [3]:
!pip install -q transformers datasets scikit-learn accelerate

In [4]:
import os
import time
import joblib
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

os.environ["WANDB_DISABLED"] = "true"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: False


In [5]:
df = pd.read_csv("combined_diseases_dataset.csv",
                 on_bad_lines="skip",
                 engine="python")
df = df[["Disease", "Symptoms"]].dropna()

print(f"Raw shape: {df.shape}")

df["Disease"] = (
    df["Disease"]
    .str.lower()
    .str.replace("-", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

df = df.drop_duplicates(subset=["Disease", "Symptoms"])

print(f"After dedup: {df.shape}")
print(f"Unique diseases before filtering: {df['Disease'].nunique()}")

Raw shape: (192113, 2)
After dedup: (192113, 2)
Unique diseases before filtering: 1683


In [6]:
MIN_SAMPLES = 100

counts = df["Disease"].value_counts()
valid_diseases = counts[counts >= MIN_SAMPLES].index
df = df[df["Disease"].isin(valid_diseases)].reset_index(drop=True)

print(f"After filtering (>= {MIN_SAMPLES} samples):")
print(f"  Rows            : {len(df):,}")
print(f"  Unique diseases : {df['Disease'].nunique()}")
print()
print("Top 10 by count:")
print(df["Disease"].value_counts().head(10))
print()
print("Bottom 10 by count:")
print(df["Disease"].value_counts().tail(10))

After filtering (>= 100 samples):
  Rows            : 179,734
  Unique diseases : 348

Top 10 by count:
Disease
hypoglycemia                      1226
pneumonia                         1223
vulvodynia                        1220
cystitis                          1220
nose disorder                     1219
complex regional pain syndrome    1218
spondylosis                       1217
conjunctivitis due to allergy     1216
vaginal cyst                      1216
esophagitis                       1216
Name: count, dtype: int64

Bottom 10 by count:
Disease
atrial flutter                 134
injury of the ankle            132
head injury                    129
arrhythmia                     124
sick sinus syndrome            121
polycystic kidney disease      120
varicose veins                 111
prostate cancer                105
peripheral arterial disease    101
coronary atherosclerosis       101
Name: count, dtype: int64


In [7]:
import random

TEMPLATES = [
    "I've been experiencing {symptoms}.",
    "I'm suffering from {symptoms}.",
    "I have been having {symptoms}.",
    "My symptoms include {symptoms}.",
    "I've noticed {symptoms} lately.",
    "I feel unwell -- I have {symptoms}.",
    "For the past few days I've had {symptoms}.",
    "I've been dealing with {symptoms} and it's really bothering me.",
    "I went to the doctor because I have {symptoms}.",
    "Can you help? I'm experiencing {symptoms}.",
    "I woke up with {symptoms}.",
    "I've been feeling really unwell with {symptoms}.",
    "Recently I started having {symptoms}.",
    "I don't feel well -- I've got {symptoms}.",
    "I need help, I'm experiencing {symptoms}.",
]

def symptoms_to_natural(symptoms_str: str, seed: int = None) -> str:
    """
    Convert a comma-separated symptom list into a natural sentence
    using a randomly chosen template.

    'fever, cough, fatigue' -> 'I've been experiencing fever, cough, and fatigue.'
    """
    if seed is not None:
        random.seed(seed)

    parts = [s.strip() for s in symptoms_str.split(",") if s.strip()]
    if len(parts) == 1:
        symptom_phrase = parts[0]
    elif len(parts) == 2:
        symptom_phrase = f"{parts[0]} and {parts[1]}"
    else:
        symptom_phrase = ", ".join(parts[:-1]) + f", and {parts[-1]}"

    template = random.choice(TEMPLATES)
    return template.format(symptoms=symptom_phrase)

In [8]:
augment_sample = df.sample(frac=0.30, random_state=42).copy()
augment_sample["Symptoms"] = [
    symptoms_to_natural(s) for s in augment_sample["Symptoms"]
]

df_final = pd.concat([df, augment_sample], ignore_index=True)

print(f"Original rows : {len(df):,}")
print(f"Augmented rows: {len(augment_sample):,}")
print(f"Total         : {len(df_final):,}")
print()
print("Sample augmented rows:")
print(df_final.tail(3)[["Disease", "Symptoms"]].to_string())

Original rows : 179,734
Augmented rows: 53,920
Total         : 233,654

Sample augmented rows:
                          Disease                                                                                                                Symptoms
233651            yeast infection  My symptoms include frequent urination, sharp abdominal pain, suprapubic pain, vaginal discharge, and vaginal itching.
233652              heart failure                                                        Recently I started having cough, sharp chest pain, and weakness.
233653  interstitial lung disease                                            I've noticed abnormal breathing sounds, cough, fever, and sleepiness lately.


In [9]:
encoder = LabelEncoder()
df_final["label"] = encoder.fit_transform(df_final["Disease"])

NUM_LABELS = len(encoder.classes_)
print(f"Number of disease classes : {NUM_LABELS}")
print(f"Sample classes            : {list(encoder.classes_[:5])}")

Number of disease classes : 348
Sample classes            : ['abdominal hernia', 'abscess of nose', 'abscess of the pharynx', 'acne', 'actinic keratosis']


In [10]:
dataset = Dataset.from_pandas(df_final[["Symptoms", "label"]])
dataset = dataset.train_test_split(test_size=0.15, seed=42)

print(f"Train : {len(dataset['train']):,} rows")
print(f"Val   : {len(dataset['test']):,} rows")

Train : 198,605 rows
Val   : 35,049 rows


In [11]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize(batch):
    tokens = tokenizer(
        batch["Symptoms"],
        padding="max_length",
        truncation=True,
        max_length=256,
    )
    tokens["labels"] = batch["label"]
    return tokens

dataset = dataset.map(tokenize, batched=True, remove_columns=["Symptoms"])
dataset["train"].set_format("torch", columns=["input_ids", "attention_mask", "labels"])
dataset["test"].set_format("torch",  columns=["input_ids", "attention_mask", "labels"])

print("Tokenization complete.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/198605 [00:00<?, ? examples/s]

Map:   0%|          | 0/35049 [00:00<?, ? examples/s]

Tokenization complete.


In [12]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=NUM_LABELS,
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params:,} parameters")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: 67,221,084 parameters


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds  = np.argmax(pred.predictions, axis=1)
    return {"accuracy": accuracy_score(labels, preds)}


OUTPUT_DIR = "/content/drive/MyDrive/my_project/checkpoints"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,
    report_to="none",
    fp16=torch.cuda.is_available(),
    optim="adamw_torch",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)
    ],
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
train_result = trainer.train()

print("\n=== Training complete ===")
print(f"Steps completed : {train_result.global_step}")
print(f"Training loss   : {train_result.training_loss:.4f}")

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss,Accuracy
500,5.744250,5.686005,0.046021
1000,5.078230,4.924896,0.228965
1500,4.117979,3.875420,0.333619
2000,3.066918,2.840688,0.456218
2500,2.186892,2.028984,0.563440
3000,1.665055,1.462747,0.675626
3500,1.328192,1.125867,0.732346
4000,1.032075,0.896802,0.768752
4500,0.874218,0.767501,0.786670
5000,0.785557,0.675138,0.800508


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


=== Training complete ===
Steps completed : 13000
Training loss   : 1.3961


In [16]:
print(trainer.state.best_model_checkpoint)
print(trainer.state.best_metric)

/content/drive/MyDrive/my_project/checkpoints/checkpoint-11500
0.8354874604125653


In [18]:
train_eval = trainer.evaluate()
print("========= Evaluation Result ===========")

for k, v in train_eval.items():
  print(f"{k}: {v:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Training Loss,Validation Loss,Step,Accuracy
0.433456,0.430583,13000,0.835487


========= Evaluation Result ===========
eval_loss: 0.4306
eval_accuracy: 0.8355


In [19]:
pred_output = trainer.predict(dataset["test"])
y_pred = np.argmax(pred_output.predictions, axis=1)
y_true = pred_output.label_ids

print(classification_report(
    y_true, y_pred, target_names=encoder.classes_, labels=list(range(min(30, NUM_LABELS))), zero_division=0
))

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


                                                 precision    recall  f1-score   support

                               abdominal hernia       0.98      0.96      0.97        46
                                abscess of nose       0.70      0.78      0.74        27
                         abscess of the pharynx       0.76      0.96      0.85        27
                                           acne       0.66      0.62      0.64        72
                              actinic keratosis       0.91      0.68      0.78       161
                            acute bronchiolitis       0.88      0.91      0.90       223
                               acute bronchitis       0.93      0.66      0.77       264
                             acute bronchospasm       0.44      0.96      0.60       147
                                 acute glaucoma       0.33      0.17      0.23        29
                            acute kidney injury       0.96      0.98      0.97       162
                    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2687: UserWarning: labels size, 30, does not match size of target_names, 348
  warnings.warn(


In [20]:
def predict_disease(text: str, top_k: int = 5) -> list:
    """
    Predict disease from a free-text symptom description.
    Returns top_k predictions sorted by confidence (highest first).
    """
    model.eval()
    device = next(model.parameters()).device

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256,
        padding=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        logits = model(**inputs).logits

    probs       = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
    top_indices = probs.argsort()[::-1][:top_k]

    return [
        {"disease": encoder.classes_[i], "confidence": round(float(probs[i]) * 100, 2)}
        for i in top_indices
    ]

In [24]:
test_inputs = [
    "I've had a really bad headache for 3 days, my body is very hot and my joints ache",
    "I feel anxious all the time, my heart races and I struggle to breathe properly",
    "My skin is really itchy with a red rash spreading across my arms",
    "I've been feeling very tired, my throat is sore and I have a blocked nose",
]

for text in test_inputs:
    print(f"Input : {text}")
    for r in predict_disease(text, top_k=7):
        print(f"  {r['disease']:<50} {r['confidence']}%")
    print()

Input : I've had a really bad headache for 3 days, my body is very hot and my joints ache
  fibromyalgia                                       9.49%
  neuralgia                                          8.8%
  chronic pain disorder                              8.41%
  muscle spasm                                       6.43%
  fracture of the rib                                5.75%
  strep throat                                       5.67%
  chronic sinusitis                                  5.32%

Input : I feel anxious all the time, my heart races and I struggle to breathe properly
  anxiety                                            9.77%
  panic disorder                                     8.72%
  paroxysmal ventricular tachycardia                 6.98%
  hyperkalemia                                       6.42%
  mitral valve disease                               6.34%
  heart failure                                      4.65%
  acute respiratory distress syndrome (ards)         3.9

In [26]:
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/my_project/final_model/tokenizer_config.json',
 '/content/drive/MyDrive/my_project/final_model/tokenizer.json')

In [27]:
joblib.dump(
    encoder,
    f"{SAVE_DIR}/label_encoder.pkl"
)

['/content/drive/MyDrive/my_project/final_model/label_encoder.pkl']

In [28]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR)
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
encoder = joblib.load(f"{SAVE_DIR}/label_encoder.pkl")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [31]:
predict_disease("I have fever, headache and body aches", top_k=5)

[{'disease': 'strep throat', 'confidence': 20.03},
 {'disease': 'abscess of the pharynx', 'confidence': 13.48},
 {'disease': 'chronic sinusitis', 'confidence': 10.15},
 {'disease': 'chronic pain disorder', 'confidence': 3.21},
 {'disease': 'acute bronchitis', 'confidence': 3.18}]